# RAO PS1-core — reviewed Colab runner
This notebook includes the reviewed Python source and sample CSVs. **No project ZIP upload needed.**
Use a fresh CPU runtime, then **Runtime → Run all**. First run uses a small, clearly labelled
synthetic PC + C + C demonstration; it is not the 54-activity public instance.
Change `DATASET` to `public-instance` for the full supplied dataset. Runtime duration is bounded;
UNKNOWN/timeouts are diagnostics, never passed schedules. This runs the Main API/worker pipeline
inside the notebook, not the React website or a public web server.

In [ ]:
DATASET = "calendar-demo"  # "calendar-demo" or "public-instance"
SCENARIOS = ["A", "B", "C"]
SECONDS_PER_SCENARIO = 30
UPLOAD_CUSTOM_CSVS = False  # True prompts for exactly the eight instance CSVs
DEMO_DATES = True  # applied ONLY to calendar-demo; public/custom get undated physical slots
assert DATASET in {"calendar-demo", "public-instance"}
assert SCENARIOS and set(SCENARIOS) <= {"A", "B", "C"}
assert isinstance(SECONDS_PER_SCENARIO, int) and SECONDS_PER_SCENARIO > 0


## 1. Restore the embedded source and fix the notebook import path

In [ ]:
import base64, hashlib, importlib, io, sys, tempfile, zipfile
from pathlib import Path
PAYLOAD = "..."
EXPECTED_SHA256 = "e25bfdc14a0afb683078c7dc50d8e0409c6645891798b1e98d358e5f9d37bec8"
raw = base64.b64decode(PAYLOAD)
assert hashlib.sha256(raw).hexdigest() == EXPECTED_SHA256
base = Path("/content") if Path("/content").exists() else Path(tempfile.gettempdir())
workspace = base / ("rao_reviewed_" + EXPECTED_SHA256[:12])
workspace.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    for entry in z.infolist():
        target = (workspace / entry.filename).resolve()
        if not target.is_relative_to(workspace.resolve()):
            raise ValueError("Unsafe archive path")
    z.extractall(workspace)
project_root = workspace / "RAO_Reviewed_Project"
backend = project_root / "backend"
assert (backend / "app" / "__init__.py").is_file()
loaded = sys.modules.get("app")
if loaded and Path(loaded.__file__).resolve() != (backend / "app" / "__init__.py").resolve():
    raise RuntimeError("Another project is already imported. Restart the runtime, then Run all.")
sys.path.insert(0, str(backend.resolve()))  # active notebook kernel, not just pip subprocess
importlib.invalidate_caches()
print("Backend:", backend)


## 2. Install and verify dependencies

In [ ]:
import subprocess
if sys.version_info < (3, 12):
    raise RuntimeError("This PS1-core project requires Python 3.12 or newer.")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(backend) + "[dev,solver]",
                "ipywidgets"], check=True)
import app
from app.modules.instance import load_instance
assert Path(app.__file__).resolve() == (backend / "app" / "__init__.py").resolve()
print("Correct app imported from:", app.__file__)
subprocess.run([sys.executable, "-c",
    "from ortools.sat.python import cp_model; print('OR-Tools import OK')"], check=True)


## 3. Select and validate the dataset

In [ ]:
instance_dir = project_root / "data" / DATASET
if UPLOAD_CUSTOM_CSVS:
    from google.colab import files
    import uuid
    supplied = files.upload()
    instance_dir = workspace / ("custom_" + uuid.uuid4().hex[:8])
    instance_dir.mkdir()
    for name, content in supplied.items():
        if Path(name).name != name or not name.lower().endswith(".csv"):
            raise ValueError("Upload CSV files only, with their original filenames.")
        (instance_dir / name).write_bytes(content)
instance = load_instance(instance_dir)
use_demo_dates = DEMO_DATES and not UPLOAD_CUSTOM_CSVS and DATASET == "calendar-demo"
print("DATASET:", instance_dir.name)
print("Activities:", len(instance.activities), "Contracts:", len(instance.contracts))
print("Synthetic demonstration dates:", use_demo_dates)
print("Public/custom schedules use physical slots until operating dates are confirmed.")


## 4. Solve → persist witness → validate → calendar/export

In [ ]:
import json, time, uuid
CALENDARS = {}
RUN_SUMMARY = {}
output_dir = workspace / ("results_" + uuid.uuid4().hex[:10])
log_path = workspace / (output_dir.name + ".log")
cmd = [sys.executable, "-u", str(project_root / "scripts" / "colab_run.py"),
       "--instance", str(instance_dir), "--output", str(output_dir),
       "--seconds", str(SECONDS_PER_SCENARIO), "--scenarios", *SCENARIOS]
if use_demo_dates:
    cmd.append("--demo-dates")
wall_limit = 180 + len(SCENARIOS) * SECONDS_PER_SCENARIO
started = time.monotonic()
timed_out = False
with log_path.open("w", encoding="utf-8") as log:
    process = subprocess.Popen(cmd, cwd=backend, stdout=log, stderr=subprocess.STDOUT)
    try:
        while True:
            try:
                process.wait(timeout=15)
                break
            except subprocess.TimeoutExpired:
                elapsed = int(time.monotonic() - started)
                print(f"Running… {elapsed}s elapsed (wall limit {wall_limit}s)", flush=True)
                if elapsed >= wall_limit:
                    timed_out = True
                    process.kill()
                    process.wait()
                    break
    except BaseException:
        process.kill()
        process.wait()
        raise
print(log_path.read_text(encoding="utf-8"))
if (output_dir / "SUMMARY.json").exists():
    RUN_SUMMARY = json.loads((output_dir / "SUMMARY.json").read_text())
    for row in RUN_SUMMARY["jobs"]:
        path = output_dir / f"CALENDAR_{row['scenario']}.json"
        if row.get("ready") and path.exists():
            value = json.loads(path.read_text())
            if value["job_id"] == row["job_id"] and value["run_id"] == RUN_SUMMARY["run_id"]:
                CALENDARS[row["scenario"]] = value
print("Validated calendars available:", list(CALENDARS))
if timed_out or process.returncode:
    print("Some/all work did not complete. Diagnostics are downloadable below. No old calendar is reused.")


## 5. Inspect the assured possession calendar
One card = one physical possession. Co-shared PC+C+C appears together. Contract-local
access indices stay separate from physical_night. Demo dates are synthetic; public/custom
data is shown by planning week and physical slot, without inventing weekdays.

In [ ]:
import html
import ipywidgets as widgets
from IPython.display import HTML, display, clear_output

def render_calendar(value, week):
    esc = lambda text: html.escape(str(text), quote=True)
    events = [e for e in value["events"] if e["week"] == week]
    parts = ["<style>.rao{font:14px system-ui;color:#19344a;background:#f4f8fb;padding:20px;border-radius:12px}.rao-grid{display:grid;grid-template-columns:repeat(auto-fit,minmax(260px,1fr));gap:12px}.rao-card{background:white;border:1px solid #cbd9e5;border-left:5px solid #197ab3;border-radius:8px;padding:16px}.rao small{color:#52677a}.rao pre{white-space:pre-wrap;overflow-wrap:anywhere;font-size:11px}</style>",
        f'<section class="rao"><h2>Scenario {esc(value["scenario"])} · Week {week}</h2>',
        f'<p>{esc(value["status"])} · validator: {esc(value["validator_authority"])} · {len(events)} possessions</p>',
        '<div class="rao-grid">']
    for e in events:
        date_label = e.get("date") or "Date not assigned"
        parts.append(f'<article class="rao-card"><small>{esc(date_label)}</small><h3>Physical night {e["physical_night"]} · {esc(" + ".join(e["access_type"]))}</h3>')
        parts.append(f'<p><b>{esc(" + ".join(e["activity_ids"]))}</b></p><p>Contracts: {esc(", ".join(e["contract_numbers"]))}</p>')
        parts.append(f'<p>Group {esc(e["co_share_group"])} · ECLO {e["eclo"]} · {esc(e["validator_status"])}</p>')
        parts.append('<details><summary>Locations, local indices & Main evidence</summary>')
        parts.append('<ul>' + ''.join(f'<li>{esc(loc)}</li>' for loc in e["location_ids"]) + '</ul>')
        parts.append('<p>Contract-local access indices: ' + esc(json.dumps(e["local_access_nights"])) + '</p>')
        parts.append('<pre>' + esc(json.dumps(e["evidence"], indent=2)) + '</pre></details></article>')
    return ''.join(parts) + '</div></section>'

if not CALENDARS:
    print("No assured calendar available. Inspect JOB/REPORT diagnostics; a timeout is not proof of infeasibility.")
else:
    scenario_picker = widgets.Dropdown(options=list(CALENDARS), description="Scenario")
    week_picker = widgets.Dropdown(description="Week")
    panel = widgets.Output()
    def draw(*_):
        with panel:
            clear_output(wait=True)
            if week_picker.value is not None:
                display(HTML(render_calendar(CALENDARS[scenario_picker.value], week_picker.value)))
    def change_scenario(*_):
        week_picker.options = sorted({e["week"] for e in CALENDARS[scenario_picker.value]["events"]})
        week_picker.value = week_picker.options[0] if week_picker.options else None
        draw()
    scenario_picker.observe(change_scenario, names="value")
    week_picker.observe(draw, names="value")
    change_scenario()
    display(widgets.HBox([scenario_picker, week_picker]), panel)


## 6. Download the result bundle
The bundle contains the persisted SQLite database, job diagnostics, witness JSON, validator
reports, calendars and submission CSV ZIPs for successful scenarios. Demo mode also exports
ICS and checks every UID. For real data, confirm dates through the full app's Calendar tab
before exporting ICS. The notebook does not turn slot indices into real-world dates.

In [ ]:
import shutil
from google.colab import files
if output_dir.exists():
    shutil.copy2(log_path, output_dir / "RUN_LOG.txt")
    download_path = shutil.make_archive(str(output_dir), "zip", output_dir.parent, output_dir.name)
    files.download(download_path)
else:
    files.download(str(log_path))
